# 🔬 03 · Geometric Optics and the Compound Microscope
### From a bent ray of light to a working microscope

**Part B · Seeing like a microscope** · Module 03 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/03_geometric_optics_and_the_compound_microscope.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib |
| **Prerequisites** | Modules 00–02 |
| **Estimated time** | 70–85 minutes |
| **Instrumentation topics covered** | Reflection/refraction and the thin-lens equation; paraxial ray tracing; the compound microscope (objective + eyepiece) and total magnification; numerical aperture, working distance and the light-gathering trade-off; Kohler illumination |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every lens, ray and illumination path in this notebook is a SIMULATED paraxial optical model; no real microscope specification sheet or CAD file is ever read.

## 🎯 Learning objectives

1. Derive and apply the thin-lens equation to find where a lens forms an image, and explain the
   difference between a real and a virtual image.
2. Trace paraxial rays through a sequence of lenses with the ABCD matrix method, and read a
   magnification and an image location directly off the resulting ray diagram.
3. Explain why a compound microscope uses two stages of magnification (objective, then eyepiece)
   rather than one single lens, and compute its total magnification.
4. Define numerical aperture (NA), explain why NA — not magnification — sets a microscope's
   light-gathering power and ultimate resolution, and describe the working-distance/NA/magnification
   trade-off across real objective lenses.
5. Explain what Kohler illumination is for, in terms of two independently-adjustable optical paths,
   and why misaligning it degrades contrast and resolution even with otherwise perfect optics.

## 🗺️ Contents

1. Reflection, refraction, and the thin-lens equation
2. Paraxial ray tracing: predicting an image before you build the microscope
3. The compound microscope: two lenses, two stages of magnification
4. Numerical aperture, working distance, and the light-gathering trade-off
5. Kohler illumination: two overlapping, independently-focused light paths

## 🧫 The biological question

Why would a lab spend fifteen thousand dollars on an objective lens barely larger than a fingertip —
and why does the answer depend entirely on *what kind of biology* that lab studies? A developmental
biologist imaging a living zebrafish embryo for six hours needs a long working distance and a
modest numerical aperture, so the objective does not crash into the dish and the sample is not
cooked by light. A structural cell biologist resolving individual nuclear pore complexes needs the
highest numerical aperture money can buy, oil immersion, and is willing to accept a working distance
of barely a tenth of a millimeter. Neither choice is about the microscope's image-analysis software
at all — it is entirely about the physical optics in front of the camera, and those optics *impose
hard limits* that no downstream computational method in this course can ever get back. Understanding
what the instrument itself can and cannot do is therefore not a detour from bioimage informatics; it
is the first constraint every other technique in this course has to live with.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
print('No GPU is needed anywhere in this notebook — everything is plain NumPy/Matplotlib.')

In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
SEED = 3
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.1; copied so this notebook runs alone) ──
import numpy as np

def thin_lens_image_distance(object_distance: float, focal_length: float) -> float:
    """The thin-lens equation solved for image distance: 1/f = 1/do + 1/di."""
    return 1.0 / (1.0 / focal_length - 1.0 / object_distance)


def ray_transfer_free_space(distance: float) -> np.ndarray:
    """The paraxial (ABCD) ray-transfer matrix for propagating a distance through free space."""
    return np.array([[1.0, distance], [0.0, 1.0]])


def ray_transfer_thin_lens(focal_length: float) -> np.ndarray:
    """The paraxial (ABCD) ray-transfer matrix for refraction by a thin lens of focal length f."""
    return np.array([[1.0, 0.0], [-1.0 / focal_length, 1.0]])


def trace_rays_through_system(rays: np.ndarray, elements: list[np.ndarray]) -> np.ndarray:
    """Propagate paraxial rays through a sequence of ABCD elements.

    ``rays`` is (n_rays, 2): each row is (height, angle_rad) at the system's entrance.
    ``elements`` is a list of 2x2 ABCD matrices (``ray_transfer_free_space`` /
    ``ray_transfer_thin_lens``), applied in order. Returns ``history`` of shape
    (len(elements) + 1, n_rays, 2): the ray state at the entrance and after each element,
    ready to plot as a ray diagram.
    """
    state = rays.copy()
    history = [state.copy()]
    for m in elements:
        state = state @ m.T
        history.append(state.copy())
    return np.array(history)


def numerical_aperture(refractive_index: float, half_angle_rad: float) -> float:
    """Numerical aperture NA = n sin(alpha), alpha the objective's half-angle of acceptance."""
    return refractive_index * np.sin(half_angle_rad)


def rayleigh_resolution(wavelength_um: float, na: float) -> float:
    """The Rayleigh resolution criterion (um): the minimum resolvable centre-to-centre distance
    between two incoherent point sources, 0.61 * wavelength / NA."""
    return 0.61 * wavelength_um / na


def abbe_resolution(wavelength_um: float, na: float) -> float:
    """Abbe's diffraction-limited resolution (um) for a periodic structure in coherent
    transmitted light: wavelength / (2 NA)."""
    return wavelength_um / (2.0 * na)

---
## 1 · Reflection, refraction, and the thin-lens equation

Module 02 introduced Snell's law, $n_i \sin\theta_i = n_t \sin\theta_t$, and the idea that a
lens is a curved piece of glass that bends light to a focus. We can now be quantitative about
*where* that focus lands. For a thin lens of focal length $f$, an object a distance $d_o$ in front of
it forms an image a distance $d_i$ behind it according to the **thin-lens equation**:

$$\frac{1}{f} = \frac{1}{d_o} + \frac{1}{d_i}$$

and the image is magnified by $m = -d_i / d_o$ (the minus sign encodes that a single-lens image of a
real object beyond the focal length is inverted).

*In words:* if $d_o > f$, light rays from the object are still diverging when they hit the lens, and
the lens bends them just enough to make them reconverge on the far side — a **real** image forms,
one you could put a screen at and see. If $d_o < f$, the lens cannot bend the rays enough to make
them converge at all; $d_i$ comes out negative, meaning the rays only *appear* to diverge from a
point behind the object — a **virtual** image, the kind a magnifying glass or an eyepiece produces,
which can only be viewed by another lens (your eye) downstream, never projected on a screen.

In [ ]:
# ── 1.1 image distance and magnification as a function of object distance ──
f = 10.0  # focal length in mm
object_distances = np.linspace(2.0, 40.0, 300)
image_distances = np.array([thin_lens_image_distance(do, f) for do in object_distances])
magnifications = -image_distances / object_distances

fig, axes = plt.subplots(1, 2, figsize=(10, 3.3))
axes[0].axvline(f, color='gray', ls='--', lw=1, label=f'focal length f={f} mm')
axes[0].plot(object_distances, image_distances, color='teal')
axes[0].set_ylim(-80, 80)
axes[0].set_xlabel('object distance $d_o$ (mm)'); axes[0].set_ylabel('image distance $d_i$ (mm)')
axes[0].set_title('Image distance vs. object distance'); axes[0].legend(fontsize=8)

axes[1].axvline(f, color='gray', ls='--', lw=1)
axes[1].plot(object_distances, magnifications, color='crimson')
axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_ylim(-10, 10)
axes[1].set_xlabel('object distance $d_o$ (mm)'); axes[1].set_ylabel('magnification $m$')
axes[1].set_title('Magnification vs. object distance')
plt.tight_layout(); plt.show()

🔍 **What just happened?** For $d_o$ just beyond $f=10$ mm, $d_i$ is large and positive and the
magnification is large and negative (a big, inverted real image, far behind the lens) — this is
exactly how a microscope **objective** is used, with the specimen placed just outside its focal
length. For $d_o < f$ (left of the dashed line), $d_i$ is negative: no real image exists on the far
side at all, only the virtual, upright, magnified image a magnifying glass or an eyepiece produces.

✏️ **Your turn.** Change `f` to `4.0` (a shorter focal length, closer to a real high-power
objective) and re-run. For a fixed object distance of, say, 4.5 mm, how does the image distance and
magnification change compared to `f=10.0`? Connect this to why objective lenses with very short focal
lengths achieve very high magnification.

---
## 2 · Paraxial ray tracing: predicting an image before you build the microscope

The thin-lens equation tells you *where* an image forms, but not what the whole bundle of
light rays from an object actually does along the way — useful once you start chaining several
lenses together, as every real microscope does. The **paraxial ray-tracing** method represents each
ray crossing a reference plane by just two numbers, $(y, \theta)$ — its height above the optical axis
and its angle to the axis (both assumed small, the "paraxial" approximation) — and represents every
optical element as a $2\times2$ **ABCD matrix** that transforms $(y, \theta)$ into the ray's state
after that element:

$$\begin{pmatrix} y' \\ \theta' \end{pmatrix} = \begin{pmatrix} A & B \\ C & D \end{pmatrix}
\begin{pmatrix} y \\ \theta \end{pmatrix}$$

Propagating a distance $d$ through free space is $\begin{pmatrix} 1 & d \\ 0 & 1 \end{pmatrix}$ (the
height changes with the angle, the angle itself is unchanged); refraction by a thin lens of focal
length $f$ is $\begin{pmatrix} 1 & 0 \\ -1/f & 1 \end{pmatrix}$ (the height is unchanged *at* the
lens, but the angle bends by an amount proportional to how far off-axis the ray is). Chaining several
elements is just matrix multiplication, in order, which is exactly what
`trace_rays_through_system` does for a whole fan of rays at once.

In [ ]:
# ── 2.1 trace a fan of rays from one object point through a single lens ──
f = 10.0
object_distance = 15.0
image_distance = thin_lens_image_distance(object_distance, f)

n_rays = 7
angles = np.linspace(-0.12, 0.12, n_rays)  # a fan of rays leaving one on-axis object point
rays0 = np.stack([np.zeros(n_rays), angles], axis=1)

elements = [ray_transfer_free_space(object_distance), ray_transfer_thin_lens(f),
            ray_transfer_free_space(image_distance)]
history = trace_rays_through_system(rays0, elements)
z_positions = np.cumsum([0, object_distance, 0, image_distance])  # z of each history row

fig, ax = plt.subplots(figsize=(8, 3.2))
for i in range(n_rays):
    ax.plot(z_positions, history[:, i, 0], color='teal', lw=1)
ax.axvline(object_distance, color='gray', ls='--', lw=1, label='lens')
ax.axvline(object_distance + image_distance, color='crimson', ls=':', lw=1.3,
           label=f'predicted image plane (d_i={image_distance:.1f} mm)')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('distance along the optical axis (mm)'); ax.set_ylabel('ray height (mm)')
ax.set_title('A fan of rays from one object point, traced through a single lens')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** Every ray in the fan — launched at different angles from the very same
object point — crosses the optical axis again at essentially the same place: the red dotted line,
which is exactly the image distance `thin_lens_image_distance` predicted analytically in Section 1.
Ray tracing and the thin-lens equation are two views of the identical underlying physics; ray tracing
simply also shows you everything in between.

✏️ **Your turn.** Add a second lens to `elements` — for instance another
`ray_transfer_thin_lens(20.0)` after a further `ray_transfer_free_space(5.0)` — and re-plot. Where do
the rays cross the axis now? This is exactly the calculation a microscope designer runs to decide how
far apart to physically mount two lenses.

---
## 3 · The compound microscope: two lenses, two stages of magnification

A single lens used as a simple magnifier (like a hand lens) is limited to perhaps 5-10x useful
magnification before aberrations (Module 05) make the image unusably blurry. A **compound
microscope** instead uses *two* lenses in series, each doing a more modest amount of work:

- The **objective** lens is placed just outside its own focal length from the specimen, forming a
  real, inverted, strongly magnified **intermediate image** partway up the microscope tube — exactly
  the "real image" case from Section 1. Its magnification is approximately
  $M_{obj} \approx L / f_{obj}$, where $L$ is the **tube length** (the objective-to-intermediate-image
  distance, standardized at 160 mm for older "finite tube length" objectives, or handled via an
  internal tube lens for modern "infinity-corrected" objectives).
- The **eyepiece** (ocular) is then placed so that this intermediate image falls just *inside* its
  focal length — the "virtual image" case from Section 1 — acting as a simple magnifier on it. Its
  magnification is $M_{eye} \approx 250\,\text{mm} / f_{eye}$, where 250 mm is the conventional "near
  point", the closest distance at which a relaxed human eye focuses comfortably.

The two stages multiply: $M_{total} = M_{obj} \times M_{eye}$. This is why a microscope's objective
and eyepiece are each labeled with their own magnification (e.g. "40x" and "10x") that you multiply
together to get the number everyone actually quotes (400x).

In [ ]:
# ── 3.1 ray-trace a full two-lens compound microscope and compute total magnification ──
tube_length = 160.0          # mm, the classical "finite tube length" standard
near_point = 250.0           # mm, conventional near point of the relaxed human eye

objective_specs = {'4x (scanning)': 4, '40x (high-dry)': 40, '100x (oil)': 100}
eyepiece_f = 25.0            # mm -> a 250/25 = 10x eyepiece

print(f"{'Objective':18s} {'f_obj (mm)':>11s} {'M_obj':>7s} {'M_eye':>7s} {'M_total':>9s}")
for name, m_obj_claimed in objective_specs.items():
    f_obj = tube_length / m_obj_claimed      # invert the nominal magnification for its focal length
    m_obj = tube_length / f_obj
    m_eye = near_point / eyepiece_f
    print(f"{name:18s} {f_obj:>11.2f} {m_obj:>7.1f} {m_eye:>7.1f} {m_obj * m_eye:>9.0f}")

# Ray-trace the 40x case end to end: object -> objective -> intermediate image -> eyepiece
f_obj = tube_length / 40
object_distance = f_obj * 1.05              # just outside the focal length, as in real use
image_distance_1 = thin_lens_image_distance(object_distance, f_obj)
n_rays = 6
angles = np.linspace(-0.1, 0.1, n_rays)
rays0 = np.stack([np.zeros(n_rays), angles], axis=1)
gap_to_eyepiece = 5.0                        # mm, intermediate image sits just inside the eyepiece's f
elements = [ray_transfer_free_space(object_distance), ray_transfer_thin_lens(f_obj),
            ray_transfer_free_space(image_distance_1 + (eyepiece_f - gap_to_eyepiece)),
            ray_transfer_thin_lens(eyepiece_f), ray_transfer_free_space(40.0)]
history = trace_rays_through_system(rays0, elements)
z = np.cumsum([0] + [getattr(m, 'd', None) or 0 for m in []])  # placeholder, z built explicitly below
z = np.cumsum([0, object_distance, 0, image_distance_1 + (eyepiece_f - gap_to_eyepiece), 0, 40.0])

fig, ax = plt.subplots(figsize=(9, 3.2))
for i in range(n_rays):
    ax.plot(z, history[:, i, 0], color='teal', lw=1)
ax.axvline(object_distance, color='gray', ls='--', lw=1, label='objective')
ax.axvline(object_distance + image_distance_1 + (eyepiece_f - gap_to_eyepiece), color='purple',
           ls='--', lw=1, label='eyepiece')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('distance along the optical axis (mm, not to scale)'); ax.set_ylabel('ray height (mm)')
ax.set_title('Full compound microscope: objective forms a real image, eyepiece magnifies it further')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The printed table shows the two per-lens magnifications multiplying
to give the familiar 40x, 400x, 1000x numbers printed on real objectives and eyepieces. In the ray
diagram, the rays cross and re-diverge sharply right after the "objective" line (the real
intermediate image), then are bent by the eyepiece into a much gentler, nearly-parallel fan — exactly
what "a virtual image viewed by a relaxed eye" should look like, since a relaxed eye focuses
comfortably on nearly-parallel rays.

✏️ **Your turn.** Swap `eyepiece_f` to `12.5` (a 20x eyepiece, $250/12.5$) and recompute the
total-magnification table. Why might a microscopist prefer a 40x objective with a 10x eyepiece (400x
total) over a 20x objective with a 20x eyepiece (also 400x total), even though the final number on
the eye is identical? (Hint: think about which lens actually has to resolve the fine structure.)

---
## 4 · Numerical aperture, working distance, and the light-gathering trade-off

Magnification only makes a blur *bigger* — it does not, by itself, reveal any more true detail.
What actually determines how much real structure an objective can deliver is its **numerical
aperture**:

$$NA = n \sin\alpha$$

where $\alpha$ is the half-angle of the cone of light the objective can accept from the specimen, and
$n$ is the refractive index of whatever sits between the specimen and the front lens (air, $n=1.00$;
water, $n=1.33$; oil, $n\approx1.51$). NA matters for two separate reasons, both consequences of
collecting a wider cone of light:

- **Light-gathering power** scales with the *solid angle* collected, which for small-to-moderate
  angles goes roughly as $NA^2$ — doubling NA gathers roughly four times the light, directly
  improving signal-to-noise in a fixed exposure time.
- **Resolution** improves too (Module 04 derives this properly): a wider acceptance cone captures
  higher spatial frequencies — finer diffracted detail — from the specimen, so resolution scales
  as $1/NA$.

Immersion media exist specifically to push NA *above* 1.0, which is otherwise impossible in air
($n=1$ already caps $\sin\alpha \le 1$): oil-immersion objectives routinely reach $NA\approx1.4$,
nearly 1.5x the maximum resolution and 2x the light-gathering of the best air objectives.

But there is no free lunch: accepting a wider cone of light from the specimen geometrically requires
getting the front lens element *closer* to the specimen — so high-NA objectives are forced to have a
very short **working distance** (the physical gap between the front lens and the specimen), often
well under 200 micrometers, which rules them out for anything thicker than a very thin sample on a
coverslip.

In [ ]:
# ── 4.1 a table of realistic objective lenses and the NA/working-distance/magnification trade-off ──
# Representative, rounded specifications (illustrative, not a single manufacturer's data sheet).
objectives = [
    dict(name='4x air, scanning', mag=4, na=0.10, wd_um=17000, medium='air', n=1.00),
    dict(name='10x air', mag=10, na=0.30, wd_um=16000, medium='air', n=1.00),
    dict(name='40x air, high-dry', mag=40, na=0.75, wd_um=660, medium='air', n=1.00),
    dict(name='60x water immersion', mag=60, na=1.20, wd_um=280, medium='water', n=1.33),
    dict(name='100x oil immersion', mag=100, na=1.40, wd_um=130, medium='oil', n=1.515),
]
wavelength_um = 0.55
print(f"{'Objective':22s} {'NA':>5s} {'WD (um)':>9s} {'Resolution (um)':>16s} {'Relative light':>14s}")
for obj in objectives:
    res = rayleigh_resolution(wavelength_um, obj['na'])
    rel_light = (obj['na'] / objectives[0]['na']) ** 2
    print(f"{obj['name']:22s} {obj['na']:>5.2f} {obj['wd_um']:>9.0f} {res:>16.3f} {rel_light:>14.1f}")

fig, ax = plt.subplots(figsize=(6, 3.3))
nas = [o['na'] for o in objectives]
wds = [o['wd_um'] for o in objectives]
ax.plot(nas, wds, 'o-', color='darkorange')
for o in objectives:
    ax.annotate(o['name'].split(',')[0].split(' immersion')[0], (o['na'], o['wd_um']), fontsize=7,
                xytext=(3, 5), textcoords='offset points')
ax.set_yscale('log')
ax.set_xlabel('numerical aperture'); ax.set_ylabel('working distance (um, log scale)')
ax.set_title('The fundamental trade-off: higher NA means shorter working distance')
plt.tight_layout(); plt.show()

🔍 **What just happened?** Resolution (in micrometers, smaller is better) improves monotonically
with NA, and relative light-gathering improves even faster (as $NA^2$) — but working distance falls
by nearly three orders of magnitude from the 4x scanning objective to the 100x oil objective. This is
precisely why the developmental biologist and the structural cell biologist from this module's
opening question reach for physically different objectives: there is no single "best" objective,
only a choice of where on this trade-off curve a given experiment needs to sit.

### 📘 Lecture notes — numerical aperture and the resolution/working-distance trade-off

**Definition.** Numerical aperture $NA = n\sin\alpha$ combines the refractive index of the immersion
medium and the half-angle of the light cone an objective accepts; it (not magnification) sets both a
microscope's light-gathering power ($\propto NA^2$) and its ultimate resolution ($\propto 1/NA$).

**Worked example.** A 60x water-immersion objective ($NA=1.2$) gathers roughly $(1.2/0.3)^2=16$ times
more light than a 10x air objective ($NA=0.3$) under the same illumination — the dominant reason
higher-magnification, higher-NA objectives are also used whenever a dim fluorescence signal, not just
fine detail, needs to be captured.

**One memorable point.** *Magnification is a choice you can always undo (crop the image, change the
eyepiece); numerical aperture is a one-way door — no downstream processing can recover resolution or
light-gathering power that a low-NA objective never collected in the first place.*

---
## 5 · Kohler illumination: two overlapping, independently-focused light paths

Even a perfect objective and eyepiece produce a poor image if the specimen is illuminated badly —
unevenly, or with the filament of the lamp itself visibly imaged onto the sample. **Kohler
illumination** (August Kohler, 1893) solves this with a deceptively simple trick: build the
illumination system so that *two separate sets of planes* are each other's conjugates, and
deliberately keep those two sets from coinciding with each other.

- **The image-forming (specimen-conjugate) path:** the **field diaphragm**, the **specimen**, the
  objective's **intermediate image**, and the eye's retina (or the camera sensor) are all images of
  one another. Closing the field diaphragm shrinks the illuminated *area* of the specimen, without
  changing its brightness or resolution at all — useful for cutting down stray light from outside
  the field of view.
- **The illumination (aperture-conjugate) path:** the **lamp filament**, the condenser's **aperture
  diaphragm**, and the objective's **back focal plane** are all images of one another — and critically,
  this set of planes does *not* coincide with the specimen-conjugate set above. Because the filament
  is imaged onto the aperture plane rather than onto the specimen, every point of the specimen
  receives light from the *entire* filament at once, averaging out its uneven, lumpy brightness into
  smooth, even illumination. Closing the *aperture* diaphragm instead narrows the cone angle of light
  illuminating the specimen — trading resolution and light throughput for increased contrast and
  depth of field (a deliberately different control from the field diaphragm).

Mixing these two controls up — a classic beginner's misalignment — either leaves the filament's shape
visible as an uneven glow across the image, or silently throws away resolution by closing the
aperture diaphragm too far "to make the image look nicer," without realizing the cost.

In [ ]:
# ── 5.1 a simplified model of the resolution/contrast trade-off when closing the aperture diaphragm ──
# A full partially-coherent image-formation calculation is well beyond this module's scope; this is a
# deliberately simplified, illustrative model of the practical trade-off every microscopist is taught.
na_objective = 0.9
wavelength_um = 0.55
condenser_opening = np.linspace(0.02, 1.0, 60)        # fraction of the condenser's maximum NA
na_illumination = condenser_opening * na_objective

# Partially-coherent resolution improves toward the (more pessimistic) incoherent limit as NA_illum grows
effective_resolution = wavelength_um / (na_objective + na_illumination)
# A simple proxy for perceived contrast: more directional (smaller NA_illum) illumination -> more contrast
relative_contrast = 1.0 / (1.0 + na_illumination / na_objective)

fig, ax1 = plt.subplots(figsize=(7, 3.3))
ax1.plot(condenser_opening, effective_resolution, color='teal')
ax1.set_xlabel('condenser aperture-diaphragm opening (fraction of max)')
ax1.set_ylabel('effective resolution (um, smaller = finer)', color='teal')
ax1.tick_params(axis='y', labelcolor='teal')
ax2 = ax1.twinx()
ax2.plot(condenser_opening, relative_contrast, color='crimson')
ax2.set_ylabel('relative contrast (simplified proxy)', color='crimson')
ax2.tick_params(axis='y', labelcolor='crimson')
ax1.set_title('Closing the aperture diaphragm: more contrast, worse resolution')
plt.tight_layout(); plt.show()

🔍 **What just happened?** As the aperture diaphragm opens (moving right), the simplified
effective-resolution curve improves (drops) while the contrast proxy falls — the classic trade-off
every introductory microscopy course teaches hands-on at the bench. This simulation is deliberately a
simplified stand-in for the full partially-coherent imaging theory (Zernike 1938; see the Further
reading), not a substitute for it — but it captures the practically important, testable consequence:
there usually is a "sweet spot" opening, often cited as roughly 70-80% of the objective's NA, rather
than wide open or fully closed.

✏️ **Your turn.** At what `condenser_opening` fraction does `na_illumination` equal
`na_objective` exactly? Microscopy textbooks commonly recommend stopping just short of this point.
Using the two curves above, explain in one sentence why "fully matched" is not actually the
recommended setting in practice.

### 📘 Lecture notes — Kohler illumination

**Definition.** Kohler illumination arranges the lamp and condenser so that two independent sets of
conjugate planes exist: a specimen-conjugate path (field diaphragm, specimen, intermediate image) and
an aperture-conjugate path (lamp filament, condenser aperture diaphragm, objective back focal plane).

**Worked example.** Closing only the field diaphragm while watching the specimen through the
eyepiece should shrink a dark boundary visibly inward around a still-evenly-lit field of view; if
instead the whole field dims unevenly or the lamp's filament shape becomes visible, the illumination
is not actually Kohler-aligned.

**One memorable point.** *Kohler illumination is "boring by design" — a correctly aligned system
should make a bright, filament-free, perfectly even field look unremarkable; that invisibility is
itself the entire point.*

## 📌 Key takeaways

- The thin-lens equation and paraxial ray tracing are two equivalent ways to predict exactly where an
  optical system forms an image — ray tracing additionally shows you the whole light path.
- A compound microscope multiplies two modest single-lens magnifications (objective, then eyepiece)
  rather than pushing one lens to do all the work.
- Numerical aperture, not magnification, sets a microscope's light-gathering power and ultimate
  resolution; higher NA always comes at the cost of a shorter working distance.
- Immersion media exist specifically to push NA above the air-imposed ceiling of 1.0.
- Kohler illumination separates illumination into two independently-controllable paths — one
  conjugate to the specimen (field diaphragm, for field-of-view size), one conjugate to the
  objective's aperture (condenser diaphragm, for the resolution/contrast trade-off).

## ✅ Checkpoint

**1.** An object sits closer to a lens than the lens's focal length. Does a real image form on the
far side? What kind of image forms instead?

<details><summary>Answer</summary>
No real image forms — `thin_lens_image_distance` returns a negative value. Only a virtual, upright,
magnified image forms, on the *same* side as the object, which can only be viewed by looking through
the lens (as with a magnifying glass or an eyepiece), never projected onto a screen.
</details>

**2.** Two objectives give the same total magnification when paired with different eyepieces. Are
they equally good for resolving fine structure? Why or why not?

<details><summary>Answer</summary>
Not necessarily — total magnification depends on both lenses, but resolution and light-gathering
depend only on the objective's numerical aperture. A lower-NA objective paired with a stronger
eyepiece can reach the same final magnification as a higher-NA objective with a weaker eyepiece, while
still resolving noticeably less true detail ("empty magnification").
</details>

**3.** You close the condenser's aperture diaphragm until the image looks sharper and more contrasty.
What have you actually traded away to get that, and which diaphragm should you have used instead if
you only wanted to reduce stray light from outside the field of view?

<details><summary>Answer</summary>
Closing the aperture diaphragm trades away resolution and light-gathering for increased contrast and
depth of field — it is not a free improvement. To cut stray light outside the viewed field without
that cost, the *field* diaphragm (not the aperture diaphragm) is the correct control.
</details>

## 🧠 Questions & discussion

- A manufacturer could in principle sell a "2000x" objective/eyepiece combination using only ordinary
  low-NA optics. Why would virtually no useful additional detail appear, no matter how high that
  number climbs? (This phenomenon has a name: "empty magnification".)
- Infinity-corrected objectives (the modern standard) route light to a separate tube lens rather than
  forming a real intermediate image at a fixed 160 mm tube length. Why might that design make it
  easier to insert extra optical components — a beamsplitter, a filter cube — into the light path
  without having to redesign every objective?
- If you were designing a microscope to image a living, intact, several-millimeter-thick organism
  versus a single layer of cultured cells on a coverslip, which of this section's objectives would you
  reach for in each case, and why?

## 🚀 Bonus — going further

- Extend the two-lens ray-tracing code in Section 3 to a three-element system that also includes a
  tube lens (as in an infinity-corrected microscope), and confirm the final image location does not
  depend on exactly where along the "infinity space" the tube lens sits.
- Compute the *depth of field* of an objective (how far the specimen can move along the optical axis
  before it visibly blurs) as a rough function of NA and magnification, and compare the 4x and 100x
  objectives from Section 4.
- Read about "parfocal" and "parcentric" objectives — look up what these terms mean mechanically, and
  explain why both properties matter when switching objectives mid-experiment on a live sample.

## 📝 Lab note

```text
Module: 03 — Geometric Optics and the Compound Microscope
Date:
An objective/eyepiece combination I would choose for MY own imaging question, and why:
One thing about Kohler illumination I would now check before starting any real imaging session:
```

## 📚 References

1. Hecht, E. (2016). *Optics* (5th ed.). Pearson.
2. Kohler, A. (1893). Ein neues Beleuchtungsverfahren fur mikrophotographische Zwecke.
   *Zeitschrift fur wissenschaftliche Mikroskopie*, 10, 433-440.
3. Davidson, M.W., Abramowitz, M. (2002). Optical microscopy. In *Encyclopedia of Imaging Science and
   Technology*. Wiley.
4. Inoue, S., Spring, K.R. (1997). *Video Microscopy: The Fundamentals* (2nd ed.). Springer.
5. Murphy, D.B., Davidson, M.W. (2012). *Fundamentals of Light Microscopy and Electronic Imaging*
   (2nd ed.). Wiley-Blackwell. — a far more complete treatment of everything in this module and the
   physics/instrumentation modules around it; recommended if you want the full optical-engineering
   depth.

## ➡️ Next

Module 04 asks what ultimately limits how small a structure any of these lenses can resolve — not as
a rule of thumb, but derived from the wave nature of light itself: diffraction, the Airy pattern, and
the point-spread function this course has been using as a black box since Module 02.